# This file will load ALL 36 CSVs ,merge into one And processed the features 
- the full dataset on which the model will be trained later 

In [ ]:
import pandas as pd
import numpy as np
import glob

- glob finds all the 36 files and returns the list of their paths
- os make sure that output folder exits before saving into it 

In [22]:
files = glob.glob("../data/raw/*.csv")
len(files)  # got the list of all 36 files

36

In [ ]:
# transferring all 36 files data to only one dataframe

new_list = []
for file in files:
    df = pd.read_csv(file)
    new_list.append(df)
df = pd.concat(new_list)
df.shape

(7274, 28)

In [24]:
df.columns.to_list()

['Period',
 'Org Code',
 'Parent Org',
 'Org name',
 'A&E attendances Type 1',
 'A&E attendances Type 2',
 'A&E attendances Other A&E Department',
 'A&E attendances Booked Appointments Type 1',
 'A&E attendances Booked Appointments Type 2',
 'A&E attendances Booked Appointments Other Department',
 'Attendances over 4hrs Type 1',
 'Attendances over 4hrs Type 2',
 'Attendances over 4hrs Other Department',
 'Attendances over 4hrs Booked Appointments Type 1',
 'Attendances over 4hrs Booked Appointments Type 2',
 'Attendances over 4hrs Booked Appointments Other Department',
 'Patients who have waited 4-12 hs from DTA to admission',
 'Patients who have waited 12+ hrs from DTA to admission',
 'Emergency admissions via A&E - Type 1',
 'Emergency admissions via A&E - Type 2',
 'Emergency admissions via A&E - Other A&E department',
 'Other emergency admissions',
 'Unnamed: 22',
 'Unnamed: 23',
 'Unnamed: 24',
 'Unnamed: 25',
 'Unnamed: 26',
 'a']

In [25]:
df.head()  # need to see what unnamed columns contain

,Period,Org Code,Parent Org,Org name,A&E attendances Type 1,A&E attendances Type 2,A&E attendances Other A&E Department,A&E attendances Booked Appointments Type 1,A&E attendances Booked Appointments Type 2,A&E attendances Booked Appointments Other Department,...,Emergency admissions via A&E - Type 1,Emergency admissions via A&E - Type 2,Emergency admissions via A&E - Other A&E department,Other emergency admissions,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,a
0,MSitAE-APRIL-2022,RLQ,NHS ENGLAND MIDLANDS,WYE VALLEY NHS TRUST,5536,0,0,139,0,0,...,1257,0,0,388,NaN,NaN,NaN,NaN,NaN,NaN
1,MSitAE-APRIL-2022,AD913,NHS ENGLAND LONDON,BECKENHAM BEACON UCC,0,0,3426,0,0,252,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,MSitAE-APRIL-2022,AJN,NHS ENGLAND NORTH EAST AND YORKSHIRE,WORKINGTON HEALTH LIMITED,0,0,312,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,MSitAE-APRIL-2022,C82010,NHS ENGLAND MIDLANDS,OAKHAM MEDICAL PRACTICE,0,0,208,0,0,0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,MSitAE-APRIL-2022,C83023,NHS ENGLAND MIDLANDS,SLEAFORD MEDICAL GROUP,0,0,509,0,0,0,...,0,0,0,9,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# "Unnamed' and "a" is of no use and should be dropped
df = df.loc[:, ~df.columns.str.contains("Unnamed")]  # df.loc[rows to keep :columns to keep]

df = df.drop(columns=["a"], errors="ignore")
df.shape

(7274, 22)

In [ ]:
# remove the type 1 hospitals whose attandance is 0
df = df[df["A&E attendances Type 1"]> 0]

df = df[df['Org name'] != 'TOTAL']

df.shape

(4433, 22)

In [28]:
# renaming the columns 
df = df.rename(columns={
    'Period': 'period',
    'Org Code': 'org_code',
    'Parent Org': 'parent_org',
    'Org name': 'org_name',
    'A&E attendances Type 1': 'type1_att',
    'A&E attendances Type 2': 'type2_att',
    'Attendances over 4hrs Type 1': 'type1_breach',
    'Attendances over 4hrs Type 2': 'type2_breach',
    'Attendances over 4hrs Other Department': 'other_breach',
    'Patients who have waited 12+ hrs from DTA to admission': 'wait_12plus',
    'Emergency admissions via A&E - Type 1': 'type1_admissions',
    'Other emergency admissions': 'other_admissions'
})

# Developing new columns that are stronger predictors

In [29]:
# finding breach_rate (out of total attendance of one hospital how many patients are treated above 4-hr timer )
df["breach_rate"] = df["type1_breach"] / df["type1_att"]

# admission_rate (out of all patients how many needed a hospital bed)
df["admission_rate"] = df["type1_admissions"] / df["type1_att"]

# severe_wait_ratio (out of all patients how many had the worst waits (12+ hours)) 
df["severe_wait_ratio"] = df["wait_12plus"] / df["type1_att"]

# breached (target variable)
df["breached"] = np.where(df["breach_rate"] > 0.30 , 1, 0)


In [30]:
# checking the new columns 
df[["breach_rate","admission_rate","severe_wait_ratio","breached"]].head(10)

,breach_rate,admission_rate,severe_wait_ratio,breached
0,0.403902,0.227059,0.048772,1
15,0.398745,0.314604,0.025670,1
16,0.363015,0.253632,0.052299,1
17,0.313460,0.247867,0.000000,1
19,0.273567,0.208694,0.000000,0
26,0.220072,0.302873,0.000200,0
27,0.621203,0.302197,0.029608,1
29,0.401058,0.298504,0.000000,1
30,0.000000,0.319923,0.000000,0
31,0.545847,0.257086,0.040493,1


In [31]:
df["breached"].value_counts()

breached
1    3755
0     678
Name: count, dtype: int64

In [32]:
# Saving the processed dataset now 
df.to_csv("../data/processed/nhs_processed.csv",index=False)
print("saved ")

saved 
